# Episode 5: Transformer Architecture - Code Walkthrough

Understanding nanoGPT by reading the code with detailed comments.

We'll walk through: Head --> MultiHeadAttention --> FeedForward --> Block --> LanguageModel

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Hyperparameters (defined globally, used by all components)
n_embd = 64      # embedding dimension - size of vectors representing each token
block_size = 32  # maximum sequence length - how far back the model can look
dropout = 0.0    # dropout rate - for regularization (0.0 = no dropout)

---

## Component 1: Head (Single Attention Head)

In [ ]:
class Head(nn.Module):
    """One head of self-attention with causal masking.
    
    This implements the Query-Key-Value attention mechanism:
    - Query: "What am I looking for?"
    - Key: "What does each position offer?"
    - Value: "What information to retrieve?"
    
    Causal masking ensures we can only attend to past positions (not future).
    """

    def __init__(self, head_size):
        super().__init__()
        
        # Three linear projections to create Q, K, V from the input
        # These are the learnable weight matrices W_Q, W_K, W_V
        # bias=False is standard in transformer implementations
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        
        # Create a lower-triangular mask (causal mask)
        # This is NOT a learnable parameter, so we use register_buffer
        # tril = [[1, 0, 0, ...],
        #         [1, 1, 0, ...],
        #         [1, 1, 1, ...], ...]
        # Ensures position i can only attend to positions <= i
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (B, T, C) where:
        #   B = batch size (number of sequences processed in parallel)
        #   T = sequence length (number of tokens in each sequence)
        #   C = embedding dimension (n_embd)
        B, T, C = x.shape
        
        # Step 1: Project input to Query, Key, Value
        # Each projection transforms (B, T, n_embd) → (B, T, head_size)
        k = self.key(x)    # Keys: "What does each position represent?"
        q = self.query(x)  # Queries: "What is each position looking for?"
        
        # Step 2: Compute attention scores (how similar is each query to each key?)
        # q @ k.transpose(-2,-1): dot product between all queries and all keys
        #   Result shape: (B, T, T) - for each query position, score for each key position
        # * C**-0.5: scale by sqrt(d_k) to prevent very large values
        #   Without scaling, softmax would saturate (all weight on one position)
        wei = q @ k.transpose(-2, -1) * C**-0.5  # (B, T, T)
        
        # Step 3: Apply causal mask
        # Set future positions to -inf (they become 0 after softmax)
        # This prevents position i from attending to positions > i
        # Critical for autoregressive generation (can't peek at future!)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # (B, T, T)
        
        # Step 4: Convert scores to probabilities
        # Softmax makes each row sum to 1 (valid probability distribution)
        # Higher scores → higher probabilities → more attention
        wei = F.softmax(wei, dim=-1)  # (B, T, T)
        wei = self.dropout(wei)  # Randomly zero out some attention weights (regularization)
        
        # Step 5: Weighted aggregation of values
        v = self.value(x)  # Values: "What information to retrieve?" (B, T, head_size)
        # wei @ v: for each position, compute weighted sum of all values
        #   (B, T, T) @ (B, T, head_size) → (B, T, head_size)
        # Result: Each position gets a context-aware representation
        #   It's a mixture of values from all past positions,
        #   weighted by how relevant each past position is (attention weights)
        out = wei @ v  # (B, T, head_size)
        return out

---

## Component 2: MultiHeadAttention

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention running in parallel.
    
    Why multiple heads?
    - Different heads can learn to focus on different types of relationships
    - Example: Head 1 learns syntax, Head 2 learns semantics, Head 3 learns long-range deps
    - Provides multiple "perspectives" on the input
    - Concatenating gives richer representation than single head
    """

    def __init__(self, num_heads, head_size):
        super().__init__()
        
        # Create multiple attention heads
        # Each head is an independent Head instance with its own parameters
        # num_heads = 4 means 4 parallel attention computations
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        
        # Projection layer after concatenation
        # Maps concatenated head outputs back to n_embd dimensions
        # This allows different heads to "talk to each other" / mix their information
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (B, T, n_embd)
        
        # Step 1: Run all heads in parallel
        # Each head processes the input independently
        # Each head output: (B, T, head_size)
        # Example with 4 heads of size 16: 4 × (B, T, 16)
        
        # Step 2: Concatenate all head outputs along the feature dimension
        # torch.cat([...], dim=-1): concatenate along last dimension
        # Result: (B, T, num_heads * head_size) = (B, T, n_embd)
        # Example: 4 heads × 16 dims = 64 dims = n_embd
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        
        # Step 3: Apply projection and dropout
        # Projection: linear transformation to mix information from different heads
        # Dropout: regularization (randomly zero out some connections during training)
        out = self.dropout(self.proj(out))
        
        return out  # (B, T, n_embd)

---

## Component 3: FeedForward

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network.
    
    A simple 2-layer MLP applied to each position independently.
    
    Purpose:
    - Attention: "communication" (positions exchange information)
    - FeedForward: "computation" (each position processes its information)
    
    Architecture: expand 4x → ReLU → contract back
    The 4x expansion is a standard design choice (more capacity for learning).
    """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # Layer 1: Expand
            # (n_embd) → (4 * n_embd)
            # Example: 64 → 256 dimensions
            # Provides more representational capacity
            nn.Linear(n_embd, 4 * n_embd),
            
            # Non-linear activation
            # Allows network to learn non-linear patterns
            # ReLU(x) = max(0, x): keeps positive values, zeros negative
            nn.ReLU(),
            
            # Layer 2: Contract
            # (4 * n_embd) → (n_embd)
            # Example: 256 → 64 dimensions
            # Projects back to original embedding dimension
            nn.Linear(4 * n_embd, n_embd),
            
            # Dropout for regularization
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x shape: (B, T, n_embd)
        # Applied independently to each position (each of the T positions)
        # Same transformation for all positions, but no mixing across positions
        # (That's what attention is for!)
        return self.net(x)  # (B, T, n_embd)

---

## Component 4: Block (Complete Transformer Block)

In [ ]:
class Block(nn.Module):
    """Transformer block: communication (attention) + computation (feed-forward).
    
    Architecture pattern (repeated in each block):
    
    Input
      ↓
    LayerNorm → MultiHeadAttention → Add residual
      ↓
    LayerNorm → FeedForward → Add residual
      ↓
    Output
    
    Key design choices:
    - Pre-norm: Layer normalization BEFORE the transformation (not after)
    - Residual connections: x = x + transformation(x)
    """

    def __init__(self, n_embd, n_head):
        super().__init__()
        
        # Calculate size of each attention head
        # Example: n_embd=64, n_head=4 → head_size=16
        # Each head gets 1/n_head of the embedding dimension
        head_size = n_embd // n_head
        
        # Multi-head self-attention: communication between positions
        self.sa = MultiHeadAttention(n_head, head_size)
        
        # Feed-forward network: computation at each position
        self.ffwd = FeedForward(n_embd)
        
        # Layer normalization: normalize activations to mean=0, variance=1
        # Stabilizes training in deep networks
        # One for before attention, one for before feed-forward
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # x shape: (B, T, n_embd)
        
        # Sub-layer 1: Multi-head attention with residual connection
        # self.ln1(x): normalize first (pre-norm architecture)
        # self.sa(...): apply multi-head attention
        # x + ...: residual connection (add input back to output)
        #
        # Residual connections allow:
        # - Gradients to flow directly through the network (helps training)
        # - Network to learn "changes" rather than complete new representations
        # - Very deep networks to be trained (without residuals, gradients vanish)
        x = x + self.sa(self.ln1(x))
        
        # Sub-layer 2: Feed-forward with residual connection
        # Same pattern: normalize → transform → add residual
        x = x + self.ffwd(self.ln2(x))
        
        return x  # (B, T, n_embd)

---

## Component 5: LanguageModel (Full Transformer)

In [ ]:
# Additional hyperparameters for the full model
vocab_size = 65  # Number of unique tokens (characters) in vocabulary
n_layer = 4      # Number of transformer blocks to stack
n_head = 4       # Number of attention heads per block
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
class LanguageModel(nn.Module):
    """Complete transformer language model.
    
    Architecture:
    1. Embeddings (token + position)
    2. Stack of transformer blocks
    3. Final layer norm
    4. Output projection to vocabulary
    
    This is the same architecture as GPT!
    GPT-3 just has more layers (96), larger embeddings (12288), and more parameters (175B).
    """

    def __init__(self):
        super().__init__()
        
        # Embedding tables (lookup tables that map indices to vectors)
        
        # Token embeddings: each token (character) gets a learnable vector
        # vocab_size rows (one per token), n_embd columns (embedding dimension)
        # Example: token 'a' (index 5) → look up row 5 → get 64-dim vector
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        
        # Position embeddings: each position (0, 1, 2, ...) gets a learnable vector
        # Why? Attention has no notion of position/order!
        # Without position embeddings: "cat dog" = "dog cat" (same representation)
        # block_size rows (one per position), n_embd columns
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        
        # Stack of transformer blocks
        # Each block: attention → feed-forward (with layer norms and residuals)
        # Deeper stacks = more layers of transformation = more powerful model
        # Example: n_layer=4 means 4 blocks stacked sequentially
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        
        # Final layer normalization
        # Stabilizes the final outputs before projection to vocabulary
        self.ln_f = nn.LayerNorm(n_embd)
        
        # Output projection: embedding space → vocabulary space
        # (n_embd) → (vocab_size)
        # Produces logits (scores) for each possible next token
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        """Forward pass: compute logits and optionally loss.
        
        Args:
            idx: input token indices, shape (B, T)
            targets: target token indices for computing loss, shape (B, T), optional
            
        Returns:
            logits: predicted scores for next token, shape (B, T, vocab_size)
            loss: cross-entropy loss if targets provided, else None
        """
        B, T = idx.shape  # B = batch size, T = sequence length

        # Step 1: Embeddings
        # Look up token embeddings for each input token
        tok_emb = self.token_embedding_table(idx)  # (B, T, n_embd)
        
        # Look up position embeddings for positions 0, 1, 2, ..., T-1
        # torch.arange(T): creates [0, 1, 2, ..., T-1]
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T, n_embd)
        
        # Combine token and position embeddings by addition
        # Broadcasting: (B, T, n_embd) + (T, n_embd) → (B, T, n_embd)
        # Now each token has both "what it is" (token) and "where it is" (position)
        x = tok_emb + pos_emb  # (B, T, n_embd)
        
        # Step 2: Pass through transformer blocks
        # Each block: attention (mix info) → feed-forward (process info)
        # Information flows and transforms through the stack
        x = self.blocks(x)  # (B, T, n_embd)
        
        # Step 3: Final layer normalization
        x = self.ln_f(x)  # (B, T, n_embd)
        
        # Step 4: Project to vocabulary
        # For each position, get scores (logits) for each possible next token
        logits = self.lm_head(x)  # (B, T, vocab_size)

        # Step 5: Compute loss if targets provided (for training)
        if targets is None:
            loss = None
        else:
            # Reshape for cross_entropy: expects (N, C) where N = batch size, C = num classes
            # Flatten batch and time dimensions: (B, T, vocab_size) → (B*T, vocab_size)
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            
            # Cross-entropy loss: measures how well predictions match targets
            # Lower loss = better predictions
            # This is what we optimize during training
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        """Generate text autoregressively.
        
        This is how we actually generate text (like ChatGPT does!):
        1. Start with some context (e.g., "Once upon a ")
        2. Predict next token
        3. Append predicted token to context
        4. Repeat
        
        Args:
            idx: starting context, shape (B, T)
            max_new_tokens: how many tokens to generate
            
        Returns:
            idx: context + generated tokens, shape (B, T + max_new_tokens)
        """
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Crop to last block_size tokens (context window limit)
            # If sequence gets longer than block_size, we can only look at the last block_size tokens
            # This is due to position embeddings only going up to block_size
            idx_cond = idx[:, -block_size:]
            
            # Get predictions for next token
            # Forward pass through the model
            logits, loss = self(idx_cond)
            
            # Focus only on the last time step (what comes after current sequence)
            # logits[:, -1, :]: for each batch, take last position, all vocab scores
            logits = logits[:, -1, :]  # becomes (B, vocab_size)
            
            # Convert logits to probabilities
            # Softmax: makes scores into a valid probability distribution (sum to 1)
            probs = F.softmax(logits, dim=-1)  # (B, vocab_size)
            
            # Sample from the distribution
            # torch.multinomial: randomly sample according to probabilities
            # Higher probability → more likely to be sampled (but not deterministic!)
            # This randomness is why you get different outputs each time
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            
            # Append sampled token to the running sequence
            # This becomes the new context for the next iteration
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        
        return idx

---

## Architecture Summary

```
LanguageModel
├── Token Embeddings (vocab_size → n_embd)
├── Position Embeddings (block_size → n_embd)
├── Transformer Blocks (×n_layer)
│   └── Block
│       ├── LayerNorm
│       ├── MultiHeadAttention (×n_head)
│       │   └── Head (Query, Key, Value, Attention)
│       ├── Residual
│       ├── LayerNorm
│       ├── FeedForward (expand 4× → ReLU → contract)
│       └── Residual
├── Final LayerNorm
└── Output Layer (n_embd → vocab_size)
```

